# Migrant Archive — Colab Transcription

Transcribe a YouTube video using **faster-whisper large-v3** on Colab's free T4 GPU.

### What this notebook does
1. Clones the project repo (or uses uploaded files)
2. Downloads audio from YouTube via `yt-dlp`
3. Transcribes using `faster-whisper` (large-v3, GPU)
4. Saves the result as JSON to your Google Drive

### Estimated time
| Video length | Transcription time on T4 GPU |
|-------------|------------------------------|
| 5 min | ~15 seconds |
| 30 min | ~2 minutes |
| 2 hours | ~8-10 minutes |

### After this notebook
Download the JSON from Drive → place in `data/raw/whisper/` → run `python backend/scripts/rebuild_index.py`

---
> Speaker diarization is disabled in this version; all segments are labelled UNKNOWN.
---

## 1. Install Dependencies

In [1]:
!pip install -q yt-dlp faster-whisper
!apt-get update -qq && apt-get install -y -qq ffmpeg curl

# Node.js v22 (NodeSource) — only used as web client fallback if cookies are present
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash -
!apt-get install -y -qq nodejs
!node --version

print("Dependencies ready.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 60.8 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
2026-06-23 20:02:54 - Installing pre-requisites
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:5 http://s

## 2. Mount Google Drive

This is where the JSON output will be saved. You'll need to authorize access.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/migrant-archive"
LOCAL_AUDIO = "/content/audio"
os.makedirs(f"{DRIVE_ROOT}/output", exist_ok=True)
os.makedirs(LOCAL_AUDIO, exist_ok=True)

print(f"Drive mounted. Output -> {DRIVE_ROOT}/output/")
print(f"Local audio cache -> {LOCAL_AUDIO}/")

Mounted at /content/drive
Drive mounted. Output -> /content/drive/MyDrive/migrant-archive/output/
Local audio cache -> /content/audio/


## 3. Get the Ingestion Code

Choose **one** method below. Option A (git clone) is recommended — it pulls all files with correct imports.

### Option A: Clone the repo

Use this if your repo is public. For private repos, use Option B.
If clone fails with a username error, switch to Option B.

In [ ]:
import sys
from pathlib import Path

# Clone the repo
REPO_URL = "https://github.com/Sebastianlopez-dev/migrant-archive.git"
!git clone {REPO_URL} /content/migrant-archive

# Add backend/core to Python path
sys.path.insert(0, str(Path("/content/migrant-archive/backend/core")))

print("Repo cloned and path configured")

### Option B: Upload files manually (private repos)

Upload these files from `backend/core/` and `data/`:
- `ingestion.py`
- `ingestion_audio.py`
- `ingestion_colab.py`
- `youtube-links.txt` (for batch — section 8)
- `cookies.txt` (optional, enables web client auth for restricted videos)

Use the Files panel (left sidebar) to upload, then run this cell:

In [3]:
# Run if using Option B (manual upload)
import sys
from pathlib import Path
sys.path.insert(0, "/content")
print("Manual upload path configured")

Manual upload path configured


## 4. Configure Your Video

Set the YouTube URL and language below. For the FILMIG channel, use `es`.
For Catalan/Spanish code-switching, try `ca` or `es`.

In [4]:
# ══════════════════════════════════════════
# EDIT THESE TWO VALUES
# ══════════════════════════════════════════

#VIDEO_URL = "https://www.youtube.com/watch?v=PASTE_VIDEO_ID_HERE"
VIDEO_URL = "https://youtu.be/mY1hw79ydY0"
LANGUAGE  = "es"       # es, en, ca, or "" for auto-detect

# ══════════════════════════════════════════
# Advanced (usually leave as-is)
# ══════════════════════════════════════════

MODEL     = "large-v3"  # tiny, base, small, medium, large-v3
DEVICE    = "cuda"      # cuda (GPU) or cpu

print(f"Video: {VIDEO_URL}")
print(f"Language: {LANGUAGE}  |  Model: {MODEL}  |  Device: {DEVICE}")

Video: https://youtu.be/mY1hw79ydY0
Language: es  |  Model: large-v3  |  Device: cuda


## 5. HuggingFace Token (legacy)

Not needed for faster-whisper. Skip this section unless you plan to switch
back to the whisperx/diarization path later.


In [ ]:
# faster-whisper does not need a HuggingFace token.
print("HF token not needed for faster-whisper — skip")

## 6. yt-dlp Setup (Two-Phase Client Strategy)

**Phase 1** — mobile clients (android → ios) WITHOUT cookies.
Mobile clients bypass YouTube's JS challenge, no runtime needed.

**Phase 2** (fallback) — web client WITH cookies + nodejs.
Only tried if Phase 1 fails AND `cookies.txt` is uploaded.

> **Export cookies from your browser:**
> ```bash
> yt-dlp --cookies-from-browser brave "https://youtube.com" \
>        --cookies cookies.txt --skip-download
> ```
> Upload `cookies.txt` via the Files panel (left sidebar).

In [5]:
import ingestion
import yt_dlp
from pathlib import Path

COOKIE_FILE = "/content/cookies.txt"
USE_COOKIES = Path(COOKIE_FILE).exists()

# ── Two-phase strategy ──
# Phase 1: mobile clients (no cookies, no JS runtime needed)
# Phase 2: web + cookies + nodejs (fallback, needs Section 1 nodejs)

PHASE1_CLIENTS = ["android", "ios"]

if USE_COOKIES:
    print(f"Cookies found → Phase 1: {PHASE1_CLIENTS} (no cookies)")
    print(f"                Phase 2: web + cookies (fallback)")
else:
    print(f"No cookies → Phase 1 only: {PHASE1_CLIENTS}")

# ── Replace _fetch_metadata ──
def _fetch_metadata_fixed(video_url: str) -> dict:
    # Phase 1 — mobile clients, no cookies, no JS runtime
    opts1: dict = {
        "quiet": True,
        "skip_download": True,
        "extractor_args": {"youtube": {"player_client": PHASE1_CLIENTS}},
    }
    try:
        with yt_dlp.YoutubeDL(opts1) as ydl:
            return ydl.extract_info(video_url, download=False)
    except Exception as e1:
        phase1_err = str(e1)

    # Phase 2 — web + cookies + nodejs (fallback)
    if USE_COOKIES:
        opts2: dict = {
            "quiet": True,
            "skip_download": True,
            "js_runtimes": {"node": {}},
            "extractor_args": {"youtube": {"player_client": ["web"]}},
            "cookiefile": COOKIE_FILE,
        }
        try:
            with yt_dlp.YoutubeDL(opts2) as ydl:
                return ydl.extract_info(video_url, download=False)
        except Exception as e2:
            raise RuntimeError(
                f"All strategies failed.\n"
                f"Phase 1 ({PHASE1_CLIENTS}): {phase1_err}\n"
                f"Phase 2 (web+cookies): {e2}"
            )
    else:
        raise RuntimeError(
            f"Phase 1 ({PHASE1_CLIENTS}) failed and no cookies for Phase 2.\n"
            f"Error: {phase1_err}\n"
            f"Upload cookies.txt for web client fallback."
        )

# ── Replace _download_audio ──
def _download_audio_fixed(video_url: str, output_dir: str = "data/audio") -> Path:
    audio_dir = Path(output_dir)
    audio_dir.mkdir(parents=True, exist_ok=True)

    info = _fetch_metadata_fixed(video_url)
    audio_path = audio_dir / f"{info['id']}.mp3"

    if audio_path.exists():
        if audio_path.stat().st_size > 1024:
            return audio_path
        audio_path.unlink()

    # Download using the strategy that worked for metadata
    # (re-run _fetch_metadata's two-phase logic inline)
    downloaded = False

    # Phase 1 — mobile clients
    opts1 = {
        "quiet": True,
        "no_warnings": True,
        "format": "bestaudio/best",
        "outtmpl": str(audio_dir / "%(id)s.%(ext)s"),
        "noplaylist": True,
        "postprocessors": [{
            "key": "FFmpegExtractAudio",
            "preferredcodec": "mp3",
            "preferredquality": "192",
        }],
        "extractor_args": {"youtube": {"player_client": PHASE1_CLIENTS}},
    }
    try:
        with yt_dlp.YoutubeDL(opts1) as ydl:
            ydl.extract_info(video_url, download=True)
        downloaded = True
    except Exception:
        pass

    # Phase 2 — web + cookies + nodejs
    if not downloaded and USE_COOKIES:
        opts2 = {
            "quiet": True,
            "no_warnings": True,
            "format": "bestaudio/best",
            "outtmpl": str(audio_dir / "%(id)s.%(ext)s"),
            "noplaylist": True,
            "postprocessors": [{
                "key": "FFmpegExtractAudio",
                "preferredcodec": "mp3",
                "preferredquality": "192",
            }],
            "js_runtimes": {"node": {}},
            "extractor_args": {"youtube": {"player_client": ["web"]}},
            "cookiefile": COOKIE_FILE,
        }
        with yt_dlp.YoutubeDL(opts2) as ydl:
            ydl.extract_info(video_url, download=True)
        downloaded = True

    # ── Verify download succeeded ──
    if not audio_path.exists() or audio_path.stat().st_size == 0:
        candidates = sorted(audio_dir.glob(f"{info['id']}.*"),
                           key=lambda p: p.stat().st_size, reverse=True)
        if candidates and candidates[0].stat().st_size > 1024:
            return candidates[0]
        raise FileNotFoundError(
            f"Download failed: {audio_path} not found. "
            f"Phases tried: {PHASE1_CLIENTS}"
            + (" + web+cookies" if USE_COOKIES else "")
            + f". Candidates: {[p.name for p in candidates]}"
        )

    return audio_path

# Monkeypatch the module
ingestion._fetch_metadata = _fetch_metadata_fixed
ingestion._download_audio = _download_audio_fixed

print("yt-dlp patched — two-phase strategy active")

Cookies found → Phase 1: ['android', 'ios'] (no cookies)
                Phase 2: web + cookies (fallback)
yt-dlp patched — two-phase strategy active


## 7. Run Transcription

This cell downloads the audio, runs Whisper, and saves the result.
For a 2-hour video, expect ~8-10 minutes on the free T4 GPU.

In [6]:
import time
import json
from faster_whisper import WhisperModel
from ingestion import VideoData, _fetch_metadata, _build_videodata, _download_audio
from ingestion_audio import _transcribe_audio

# ── Step 1: Fetch metadata ────────────────────────────────────────────
print("Fetching video metadata ...")
info = _fetch_metadata(VIDEO_URL)
video_id = info["id"]
title = info.get("title", "Unknown")
duration_mins = info.get("duration", 0) / 60

print(f"   Title: {title}")
print(f"   Duration: {duration_mins:.0f} min")
print(f"   Channel: {info.get('channel', 'Unknown')}")

# ── Step 2: Download audio ────────────────────────────────────────────
print(f"\nDownloading audio ...")
t0 = time.time()
audio_path = _download_audio(VIDEO_URL, output_dir=LOCAL_AUDIO)
print(f"   Done in {time.time() - t0:.0f}s -> {audio_path}")

# ── Verify audio file exists before proceeding ──
if not audio_path.exists() or audio_path.stat().st_size < 1024:
    raise FileNotFoundError(
        f"Audio file missing or too small: {audio_path}\n"
        "Check Section 6 (yt-dlp Setup) ran successfully.\n"
        "Try running Section 6 again, then re-run this cell."
    )
print(f"   Verified: {audio_path.stat().st_size / 1024:.0f} KB")

# ── Step 3: Transcribe ────────────────────────────────────────────────
print(f"\nTranscribing with faster-whisper ({MODEL}, {DEVICE}) ...")
print(f"    This may take a few minutes for long videos ...")
t0 = time.time()

segments = _transcribe_audio(
    audio_path=str(audio_path),
    language=LANGUAGE,
    model_size=MODEL,
    device=DEVICE,
)

elapsed = time.time() - t0
rtf = elapsed / (info.get("duration", 1) / 60)  # real-time factor
print(f"   Done in {elapsed:.0f}s ({elapsed/60:.1f} min)")
print(f"   Speed: {rtf:.1f}x real-time")
print(f"   Segments: {len(segments)}")

# ── Step 4: Build VideoData & save ────────────────────────────────────
video_data = _build_videodata(info, segments)
output_path = video_data.save_json(output_dir=f"{DRIVE_ROOT}/output")

print(f"\nSaved to Drive: {output_path}")
print(f"   File size: {output_path.stat().st_size / 1024:.0f} KB")

# ── Step 5: Preview ───────────────────────────────────────────────────
print(f"\n{'─'*60}")
print(f"PREVIEW (first 300 chars):")
print(f"{'─'*60}")
print(video_data.full_text[:100])
print(f"{'─'*60}")

# ── Step 6: Summary ───────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"TRANSCRIPTION COMPLETE")
print(f"{'='*60}")
print(f"Video:    {title}")
print(f"Duration: {duration_mins:.0f} min")
print(f"Segments: {len(segments)}")
print(f"Chars:    {len(video_data.full_text):,}")
print(f"Speed:    {rtf:.1f}x real-time")
print(f"Saved:    {output_path}")
print(f"\n Next: Download this JSON → place in data/raw/whisper/ → rebuild index")

Fetching video metadata ...


   Title: Mujeres del Maíz: relatos migrantes escritos por 12 mujeres latinoamericanas
   Duration: 13 min
   Channel: Plataforma Cero



   Done in 27s -> /content/audio/mY1hw79ydY0.mp3
   Verified: 18704 KB

Transcribing with faster-whisper (large-v3, cuda) ...
    This may take a few minutes for long videos ...
   Done in 153s (2.5 min)
   Speed: 11.5x real-time
   Segments: 125

Saved to Drive: /content/drive/MyDrive/migrant-archive/output/mY1hw79ydY0.json
   File size: 628 KB

────────────────────────────────────────────────────────────
PREVIEW (first 300 chars):
────────────────────────────────────────────────────────────
Title: Mujeres del Maíz: relatos migrantes escritos por 12 mujeres latinoamericanas
Description: Ten
────────────────────────────────────────────────────────────

TRANSCRIPTION COMPLETE
Video:    Mujeres del Maíz: relatos migrantes escritos por 12 mujeres latinoamericanas
Duration: 13 min
Segments: 125
Chars:    11,822
Speed:    11.5x real-time
Saved:    /content/drive/MyDrive/migrant-archive/output/mY1hw79ydY0.json

 Next: Download this JSON → place in data/raw/whisper/ → rebuild index


---

## 8. Batch Processing (Multiple Videos)

Upload `youtube-links.txt` to Colab (Files panel -> upload), then run this cell.
It reads all video URLs from the file and processes them one by one.

**Already-processed videos are skipped** -- add new URLs to the file and
re-run safely without re-transcribing everything.

> Use this for the first bulk run. Use **Section 7** for a single new video.

In [7]:
import time
import os
from pathlib import Path
from ingestion import _fetch_metadata
from ingestion_audio import extract_single_video

# ══════════════════════════════════════════
# CONFIGURE
# ══════════════════════════════════════════
LINKS_FILE = "/content/youtube-links.txt"   # uploaded via Files panel
LANGUAGE   = "es"
MODEL      = "large-v3"
DEVICE     = "cuda"
OUTPUT_DIR = f"{DRIVE_ROOT}/output"
AUDIO_DIR  = LOCAL_AUDIO
SLEEP_SEC  = 45   # delay between videos to avoid rate-limit

# ══════════════════════════════════════════
# Read links file (skip comments)
# ══════════════════════════════════════════
raw = Path(LINKS_FILE).read_text(encoding="utf-8").splitlines()
urls = [line.strip() for line in raw if line.strip() and not line.strip().startswith("#")]

print(f"{len(urls)} video(s) in links file")

# ══════════════════════════════════════════
# Process each video
# ══════════════════════════════════════════
ok, skipped, failed = 0, 0, 0

for i, url in enumerate(urls, 1):
    print(f"\n{'═'*60}")
    print(f"[{i}/{len(urls)}] {url}")
    print(f"{'═'*60}")

    # ── Check if already processed ──
    try:
        info = _fetch_metadata(url)
        video_id = info["id"]
    except Exception as e:
        print(f"   [ERROR] Metadata fetch failed: {e}")
        failed += 1
        continue

    json_path = Path(OUTPUT_DIR) / f"{video_id}.json"
    if json_path.exists():
        print(f"   [SKIP] Already exists: {json_path.name}")
        skipped += 1
        continue

    # ── Transcribe ──
    print(f"   Title: {info.get('title', 'Unknown')}")
    print(f"   Duration: {info.get('duration', 0) / 60:.0f} min")
    print(f"   Transcribing ...")

    t0 = time.time()
    try:
        data = extract_single_video(
            video_url=url,
            languages=[LANGUAGE],
            model_size=MODEL,
            device=DEVICE,
            output_dir=OUTPUT_DIR,
            audio_dir=AUDIO_DIR,
        )
        saved = data.save_json(output_dir=OUTPUT_DIR)
        elapsed = time.time() - t0
        print(f"   [OK] {len(data.transcript_segments)} segments in {elapsed:.0f}s")
        print(f"        Saved: {saved}")
        ok += 1
    except Exception as e:
        print(f"   [ERROR] Failed: {e}")
        failed += 1

    print(f"   Sleeping {SLEEP_SEC}s to avoid rate-limit ...")
    time.sleep(SLEEP_SEC)

# ══════════════════════════════════════════
# Summary
# ══════════════════════════════════════════
print(f"\n{'='*60}")
print(f"BATCH COMPLETE")
print(f"{'='*60}")
print(f"  [OK]    Processed: {ok}")
print(f"  [SKIP]  Skipped:   {skipped}")
print(f"  [ERROR] Failed:    {failed}")
print(f"\n  Output: {OUTPUT_DIR}/")
print(f"\n  Next: Download JSONs from Drive -> place in data/raw/whisper/ -> rebuild index")


10 video(s) in links file

════════════════════════════════════════════════════════════
[1/10] https://youtu.be/APgxfNssxGQ
════════════════════════════════════════════════════════════


   [SKIP] Already exists: APgxfNssxGQ.json

════════════════════════════════════════════════════════════
[2/10] https://youtu.be/myxPJCDedOE
════════════════════════════════════════════════════════════


   [SKIP] Already exists: myxPJCDedOE.json

════════════════════════════════════════════════════════════
[3/10] https://youtu.be/CTmWjuQcvHY
════════════════════════════════════════════════════════════


   [SKIP] Already exists: CTmWjuQcvHY.json

════════════════════════════════════════════════════════════
[4/10] https://youtu.be/AMn2LdbJ_A4
════════════════════════════════════════════════════════════


   [SKIP] Already exists: AMn2LdbJ_A4.json

════════════════════════════════════════════════════════════
[5/10] https://youtu.be/i2RdECeyHEE
════════════════════════════════════════════════════════════


...

   [SKIP] Already exists: VJqe2h0U1Fs.json

════════════════════════════════════════════════════════════
[7/10] https://youtu.be/TB66kk0_qpI
════════════════════════════════════════════════════════════


   [SKIP] Already exists: TB66kk0_qpI.json

════════════════════════════════════════════════════════════
[8/10] https://youtu.be/Jc1xJ4V4xU4
════════════════════════════════════════════════════════════


   [SKIP] Already exists: Jc1xJ4V4xU4.json

════════════════════════════════════════════════════════════
[9/10] https://youtu.be/FKymj4_fn3g
════════════════════════════════════════════════════════════


   [SKIP] Already exists: FKymj4_fn3g.json

════════════════════════════════════════════════════════════
[10/10] https://youtu.be/mY1hw79ydY0
════════════════════════════════════════════════════════════


   [SKIP] Already exists: mY1hw79ydY0.json

BATCH COMPLETE
  [OK]    Processed: 0
  [SKIP]  Skipped:   10
  [ERROR] Failed:    0

  Output: /content/drive/MyDrive/migrant-archive/output/

  Next: Download JSONs from Drive -> place in data/raw/whisper/ -> rebuild index


## 9. Download the Result

The JSON is saved in your Google Drive at `migrant-archive/output/{video_id}.json`.

**Option A**: Download directly from [drive.google.com](https://drive.google.com) → My Drive → migrant-archive → output

**Option B**: Run this cell to download via Colab:

In [9]:
from google.colab import files

# Download the JSON file to your local machine
files.download(str(OUTPUT_DIR))

print("\nPlace this file in: migrant-archive/data/raw/whisper/")
print("Then run: python backend/scripts/rebuild_index.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Place this file in: migrant-archive/data/raw/whisper/
Then run: python backend/scripts/rebuild_index.py


---

## 10. Troubleshooting

| Problem | Likely cause | Fix |
|---------|-------------|-----|
| `No such file or directory` on mp3 | yt-dlp downloaded but path mismatch, or all clients failed | Delete `/content/audio/*`, re-run Section 6, then Section 7. If it persists, add `cookies.txt`. |
| `n challenge solving failed` | Mobile clients blocked — YouTube may have rotated | Re-export `cookies.txt` from your browser, upload it, re-run Section 6. |
| `Sign in to confirm you're not a bot` | YouTube blocked Colab's IP — needs auth | Upload `cookies.txt` from your browser, re-run Section 6 + 7. |
| `Audio file missing or too small` | Download silently failed — cookies expired or client blocked | Re-export fresh `cookies.txt`, upload, re-run Section 6 + 7. |
| `faster_whisper` import error | Colab lost packages between sessions | Re-run Section 1. If persistent: Runtime → Restart runtime. |
| `yt-dlp` says video unavailable | Video is private, age-restricted, or region-blocked | Try another URL, or upload browser cookies. |
| `No module named 'ingestion'` | Section 3 clone failed (private repo?) | Use Option B: upload files manually, then run that cell. |
| Out of memory (OOM) | large-v3 needs ~10GB VRAM on long videos | Switch `MODEL = "medium"` — still excellent quality, less RAM. |
| GPU not available | Colab T4 quota exhausted | Wait a few hours or use `DEVICE = "cpu"` (much slower). |
| `ModuleNotFoundError: No module named 'faster_whisper'` | Section 1 did not install faster-whisper | Re-run Section 1, then Runtime → Restart runtime. |
| Drive mount fails | Auth timeout or wrong account | Re-run Section 2. Make sure you're logged into the correct Google account. |
| File downloads as 0 bytes | Stale corrupted cache from previous failed run | Run `!rm -f /content/audio/* /content/drive/MyDrive/migrant-archive/audio/*`, then re-run. |